## 2단계 Multi Chain 만들기 - 여행지 정보 시스템

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [2]:
from langchain_openai import ChatOpenAI

# 기본 LLM 제공자: "groq" 또는 "openai" — get_llm() 호출 시 provider 를 생략하면 이 값을 사용
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

In [4]:
llm = get_llm(temperature=0.7, seed=42, provider="groq")  # 제공자를 직접 지정 (키/모델명은 공통 설정 셀의 get_llm 사용)

[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [8]:
from langchain_core.runnables.passthrough import RunnablePassthrough
from langchain_core.output_parsers.string import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# 1단계: 여행지 → 대표 명소 1개 추천
## 직업 -> 해야할 일 -> 답변 형태 -> 예시
recommend_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 세계 여행지를 잘 아는 여행 전문가입니다. "
     "사용자가 도시나 국가를 입력하면, 그 지역의 가장 대표적인 관광 명소 딱 1곳을 추천합니다. "
     "설명이나 인사말 없이 명소 이름만 한 줄로 답하세요. "
     "예시) 입력: 파리 → 출력: 에펠탑"),
    ("user", "여행지: {destination}"),
])

# 2단계: 명소 → 상세 정보 제공
detail_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 친절한 여행 가이드입니다. "
     "사용자가 관광 명소 이름을 알려주면, 여행자에게 도움이 되도록 아래 항목을 정리해 설명합니다.\n"
     "1. 명소 소개 (어떤 곳인지)\n"
     "2. 역사와 의미\n"
     "3. 볼거리와 즐길 거리\n"
     "4. 방문 팁 (추천 시간대, 주의사항 등)\n"
     "정확하지 않은 정보(입장료, 운영시간 등)는 지어내지 말고, 확인이 필요하다고 안내하세요."),
    ("user", "관광 명소: {attraction}"),
])

chat_prompt = ChatPromptTemplate.from_messages([
    recommend_prompt,
    detail_prompt
])

chain1 = recommend_prompt | llm | StrOutputParser()   # 출력: "에펠탑"
chain2 = detail_prompt | llm | StrOutputParser()      # 출력: 상세 설명 글

full_chain = (
    RunnablePassthrough.assign(attraction=chain1)   # 1단계 결과를 attraction에 저장
    | RunnablePassthrough.assign(detail=chain2)     # attraction으로 2단계 실행 → detail에 저장
)

# excution
result = full_chain.invoke({"destination" : "일본"})

print("📍 여행지:", result["destination"])
print("⭐ 추천 명소:", result["attraction"])
print("\n📖 상세 정보\n", result["detail"])

📍 여행지: 일본
⭐ 추천 명소: 후지산

📖 상세 정보
 **후지산 (Mount Fuji) 여행 가이드**

---

### 1. 명소 소개  
후지산은 일본 혼슈 중앙에 위치한 활화산으로, 높이 3,776 m에 달해 일본 최고봉입니다. ‘후지’라는 이름은 ‘불멸(不死)’ 혹은 ‘불(富)’과 ‘자(士)’가 결합된 뜻이라는 설이 있으며, 그 우뚝한 원뿔형 실루엣은 세계적으로도 유명합니다. 일본을 대표하는 상징이자, 사진·회화·시와 노래 등 문화 전반에 영감을 주는 존재입니다.

---

### 2. 역사와 의미  
- **신화와 종교**: 오래전부터 후지산은 신성한 산으로 여겨졌습니다. 신도와 불교에서 각각 ‘후지산신(富士山神)’과 ‘산신(산의 신)’을 모시며, 등산 자체가 영적 수련으로 인식되었습니다.  
- **문화 유산**: 2013년 유네스코 세계문화유산에 등재되었으며, ‘후지산 회화·문학·시’ 등 다양한 예술작품에 영감을 주었습니다.  
- **역사적 사건**: 에도 시대에는 ‘후지산 신사’를 중심으로 순례가 활발했으며, 메이지 이후에는 등산과 관광이 대중화되었습니다.  

---

### 3. 볼거리와 즐길 거리  

| 구역·코스 | 주요 볼거리·활동 | 비고 |
|-----------|----------------|------|
| **후지 5호(5th Station)** – 케이스케와 고텐바시 등 | 산 중턱에서 보는 파노라마 풍경, 기념품 가게, 카페 | 등산 전 준비와 휴식에 최적 |
| **산 정상(구라시라코우)** | 일출(‘가오리’), 일몰, 구름 위를 걷는 듯한 풍경 | 등산 시즌(보통 7~9월)에는 사전 예약이 필요할 수 있음 |
| **호수 주변(후지오코, 가와구치코 등)** | 호수와 산이 어우러진 풍경, 보트 타기, 온천 | 사진 명소와 카페가 많아 여유로운 산책 가능 |
| **후지산 문화센터·미술관** | 후지산 관련 전시, 전통 공예 체험 | 문화와 역사를 동시에 체험 |
| **산악 트레일** | 초보자용 ‘요시다 코스